In [2]:
#####################################################
# imports
#####################################################

import numpy as np
import os
import pandas as pd
import seaborn as sns
import sys

# Note that suppressing warnings using the "warnings" package does not work when using multiprocessing
# see https://github.com/scikit-learn/scikit-learn/issues/12939
import warnings

warnings.filterwarnings("ignore")

# This works for most things, but obviously removes potentially important warnings
os.environ["PYTHONWARNINGS"] = "ignore"
# This redirects all output to standard error into a file, which at least makes the output clean
# sys.stderr = open('log.txt', 'w')

from collections import ChainMap, defaultdict

from dotenv import load_dotenv

from imblearn.over_sampling import ADASYN, SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline

from joblib import Parallel, delayed

from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from mlxtend.plotting import plot_sequential_feature_selection as plot_sfs

from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform
from scipy.stats import spearmanr
from scipy.cluster.hierarchy import dendrogram

from sklearn import set_config
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, RandomForestRegressor
from sklearn.feature_selection import SelectFromModel, mutual_info_classif, RFECV, SequentialFeatureSelector
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression, RidgeCV, RidgeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, \
    recall_score, silhouette_score
from sklearn.model_selection import StratifiedKFold, train_test_split, RandomizedSearchCV, GridSearchCV, \
    RepeatedStratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.cluster import FeatureAgglomeration, KMeans

from skopt import BayesSearchCV
from skopt.space import Categorical, Integer, Real

from timeit import default_timer as timer

from xgboost import XGBClassifier

import matplotlib.colors as colors
import matplotlib.pyplot as plt

# make transform operations return dataframes
set_config(transform_output="pandas")

# remove dtype from printing
np.set_printoptions(legacy='1.25')


In [3]:
#####################################################
# Experimental Parameters
#
#####################################################
# TODO this should be a config object that is saved with the results

# get variables from .env, e.g. DATA_DIR
load_dotenv()


def min_increase_for_progression(edss_score):
    return 1.0 if edss_score <= 5 else 0.5


# how progression is calculated
# Constraint that the minimum previous EDSS score is 3.0, given that was the minimum used for recruitment.
base_scores = [3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 6.5]

base_scores_subset_type = "all"
base_scores_subset = base_scores
if base_scores_subset_type == "low":
    base_scores_subset = [3.0, 3.5, 4.0]
elif base_scores_subset_type == "medium":
    base_scores_subset = [4.5, 5.0]
elif base_scores_subset_type == "high":
    base_scores_subset = [5.5, 6.0, 6.5]

min_progression_points = 2  # needs to be min 2 to have confirmed progression
min_progression_at_all_points = True  # if False can be at any timepoint
min_progression_at_remaining_points = False  # min progression must remain for all timepoints
# TODO the following parameter should probably be a clinically significant difference, i.e. a difference of >= 1.0 for <5, so if the EDSS at the point before progression is 4.0, previous values would have to be <=4.5, i.e. 4.5 4.5 4.0 5.0 5.0 is OK.
all_previous_points_below_threshold = False

# Alternatively use participant progression
use_transition_progression = False

# feature selection
mi_threshold = 0.0
feature_selection_type = "sfs"
show_feature_analysis = False
show_mi_feature_selection = False
show_cluster_feature_selection = False
show_pca_feature_selection = False
show_rfe_feature_selection = False
show_sfs_feature_selection = False
show_feature_agglomeration = False
show_redundant_features = False

drop_DMO_features = False
drop_non_DMO_features = False
drop_features_as_predictors = ['EDFSCR1L']
only_include_features = ['ws_30_var_w', 'time_since_onset', 'cadence_all_avg_w',
                         'n_days_w', 'strlen_30_var_w', 'strlen_1030_avg_w']

# number of parallel jobs to run
# if n_jobs <0, the actual n_jobs is 1 + (n_cpus + n_jobs)
fs_n_jobs = 10

# under/over sampling
do_data_sampling = False

random_state = 42
scoring = 'recall_macro'

# hyper parameter tuning values
do_svm_hpt_test = True
do_knn_hpt_test = True
do_mlp_hpt_test = True
do_xgb_hpt_test = True

hpt_n_iter = 2  # set to 2 for BayesSearchCV as higher values produce (erratic) worse results
hpt_verbose = 0
hpt_n_jobs = 5
hpt_cv = 5

# cross validation
cv_folds = 5

show_data_distributions = False


In [4]:
#####################################################
# Load data
#
#####################################################

data_dir = os.getenv("DATA_DIR", "Dataset/")

# read in main MS dataset
data_file_path = data_dir + r"MS_dataset_v.7.3.csv"
data = pd.read_csv(data_file_path)
data['visit.number'] = data['visit.number'].str.upper()  # for consistency with filenames

# get participant demographics
demo_data = data.copy()
demo_data = demo_data[demo_data["visit.number"] == "T1"]
print()

# after getting table values, just crop to the necessary columns
cols_from_dataset = ['Local.Participant', 'EDFSCR1L', 'visit.number',
                     'age', 'gender', 'mstypeonset', 'mstypediag', 'visit.date']
data = data[cols_from_dataset]

# forward fill within groups to ensure no leakage of demographics between participants
cols_from_dataset = [col for col in cols_from_dataset if col in ['age', 'gender', 'mstypeonset', 'mstypediag']]
for col in cols_from_dataset:
    data[col] = data.groupby('Local.Participant')[col].ffill()

# we need to get the time difference and convert from timedelta to something that the ML models can use - probably unix time
data['time_since_diag'] = (pd.to_datetime(data['visit.date']) - pd.to_datetime(data['mstypediag'])).dt.total_seconds()
data['time_since_onset'] = (pd.to_datetime(data['visit.date']) - pd.to_datetime(data['mstypeonset'])).dt.total_seconds()

# drop cols used for calculation
data = data.drop(['mstypeonset', 'mstypediag', 'visit.date'], axis=1)

# convert gender to 0 and 1
data['gender'] = data['gender'].map({'Male': 0, 'Female': 1})

# print(f'Column NaN counts out of {len(data)} rows')
# print(data.isna().sum())
print(f'Nan counts at each timepoint')
g = data.groupby('visit.number')
print(g.count().rsub(g.size(), axis=0))

# drop rows with no EDS score (i.e. EDFSCR1L)
count_before = data["Local.Participant"].nunique()
print(f"Dropping {sum(data['EDFSCR1L'].isna())} rows with no EDSS score...")
data = data.dropna(subset=['EDFSCR1L'])
print(
    f"Dropping rows results in {count_before - data["Local.Participant"].nunique()} participants excluded with no EDSS scores...")

print(f'Nan counts at each timepoint')
g = data.groupby('visit.number')
print(g.count().rsub(g.size(), axis=0))

# Crop down to just the necessary columns
data = data.sort_values(by=['Local.Participant', 'visit.number'])



Nan counts at each timepoint
              Local.Participant  EDFSCR1L  age  gender  time_since_diag  \
visit.number                                                              
T1                            0         4    0       0                2   
T2                            0       105    0       0               62   
T3                            0       121    0       0               87   
T4                            0       147    0       0              107   
T5                            0       112    0       0               71   

              time_since_onset  
visit.number                    
T1                           2  
T2                          62  
T3                          87  
T4                         107  
T5                          71  
Dropping 489 rows with no EDSS score...
Dropping rows results in 0 participants excluded with no EDSS scores...
Nan counts at each timepoint
              Local.Participant  EDFSCR1L  age  gender  time_since_diag 

In [5]:
#####################################################
# Functions to calculate participant-based progression
#
#####################################################

timepoints = ['T1', 'T2', 'T3', 'T4', 'T5']


def is_progression(tp_scores, i, tp0, min_progression, progression_point):
    if (tp_scores[i + progression_point] - tp_scores[tp0] >= min_progression):
        if progression_point < min_progression_points:
            return is_progression(tp_scores, i, tp0, min_progression, progression_point + 1)
        return True
    return False


def calc_progression(tp_scores, min_score):
    first_valid_timepoint = None

    for i, tp0 in enumerate(timepoints):
        if i == len(tp_scores) - min_progression_points:
            break
        if tp_scores[tp0] >= min_score:
            if not first_valid_timepoint:
                first_valid_timepoint = tp0
            min_progression = min_increase_for_progression(tp_scores[tp0])
            if (is_progression(tp_scores, i, tp0, min_progression, 1)):
                return {'timepoint': tp0, 'progression': 1}

    return {'timepoint': first_valid_timepoint, 'progression': 0}


# get progression label column
def threshold_progress_point(tp_scores, thresholds):
    first_valid_timepoint = None

    for threshold in thresholds:
        t1_ge_threshold = tp_scores['T1'] >= threshold
        t2_ge_threshold = tp_scores['T2'] >= threshold
        t3_ge_threshold = tp_scores['T3'] >= threshold
        t4_ge_threshold = tp_scores['T4'] >= threshold
        t5_ge_threshold = tp_scores['T5'] >= threshold

        # only consider two timepoint progression
        base_t = 'T1'
        if threshold > tp_scores[base_t] >= thresholds[0]:
            first_valid_timepoint = base_t
            min_progression = min_increase_for_progression(tp_scores[base_t])
            t2_progression = t2_ge_threshold and (tp_scores['T2'] - tp_scores[base_t]) >= min_progression
            t3_progression = t3_ge_threshold and (tp_scores['T3'] - tp_scores[base_t]) >= min_progression
            t4_progression = t4_ge_threshold and (tp_scores['T4'] - tp_scores[base_t]) >= min_progression
            t5_progression = t5_ge_threshold and (tp_scores['T5'] - tp_scores[base_t]) >= min_progression

            if min_progression_at_remaining_points:
                if min_progression_at_all_points:
                    if t2_progression and t3_progression and t4_progression and t5_progression:
                        return {'timepoint': base_t, 'progression': 1}
                else:
                    if t2_ge_threshold and t3_ge_threshold and t4_ge_threshold and t5_ge_threshold:
                        if t2_progression or t3_progression or t4_progression or t5_progression:
                            return {'timepoint': base_t, 'progression': 1}

            else:
                if min_progression_points == 1:
                    if t2_progression:
                        return {'timepoint': base_t, 'progression': 1}
                elif min_progression_points == 2:
                    if min_progression_at_all_points:
                        if t2_progression and t3_progression:
                            return {'timepoint': base_t, 'progression': 1}
                    elif t2_ge_threshold and t3_ge_threshold:
                        if t2_progression or t3_progression:
                            return {'timepoint': base_t, 'progression': 1}
                elif min_progression_points == 3:
                    if min_progression_at_all_points:
                        if t2_progression and t3_progression and t4_progression:
                            return {'timepoint': base_t, 'progression': 1}
                    elif t2_ge_threshold and t3_ge_threshold and t4_ge_threshold:
                        if t2_progression or t3_progression or t4_progression:
                            return {'timepoint': base_t, 'progression': 1}
                elif min_progression_points == 4:
                    if min_progression_at_all_points:
                        if t2_progression and t3_progression and t4_progression and t5_progression:
                            return {'timepoint': base_t, 'progression': 1}
                    elif t2_ge_threshold and t3_ge_threshold and t4_ge_threshold and t5_ge_threshold:
                        if t2_progression or t3_progression or t4_progression or t5_progression:
                            return {'timepoint': base_t, 'progression': 1}

        if min_progression_points < 4:
            if not all_previous_points_below_threshold or not t1_ge_threshold:
                base_t = 'T2'
                min_progression = min_increase_for_progression(tp_scores[base_t])
                if threshold > tp_scores[base_t] >= thresholds[0]:
                    if not first_valid_timepoint:
                        first_valid_timepoint = base_t
                    t3_progression = t3_ge_threshold and (tp_scores['T3'] - tp_scores[base_t]) >= min_progression
                    t4_progression = t4_ge_threshold and (tp_scores['T4'] - tp_scores[base_t]) >= min_progression
                    t5_progression = t5_ge_threshold and (tp_scores['T5'] - tp_scores[base_t]) >= min_progression

                    if min_progression_at_remaining_points:
                        if min_progression_at_all_points:
                            if t3_progression and t4_progression and t5_progression:
                                return {'timepoint': base_t, 'progression': 1}
                        else:
                            if t3_ge_threshold and t4_ge_threshold and t5_ge_threshold:
                                if t3_progression or t4_progression or t5_progression:
                                    return {'timepoint': base_t, 'progression': 1}

                    else:
                        if min_progression_points == 1:
                            if t3_progression:
                                return {'timepoint': base_t, 'progression': 1}
                        elif min_progression_points == 2:
                            if min_progression_at_all_points:
                                if t3_progression and t4_progression:
                                    return {'timepoint': base_t, 'progression': 1}
                            elif t3_ge_threshold or t4_ge_threshold:
                                if t3_progression or t4_progression:
                                    return {'timepoint': base_t, 'progression': 1}
                        elif min_progression_points == 3:
                            if min_progression_at_all_points:
                                if t3_progression and t4_progression and t5_progression:
                                    return {'timepoint': base_t, 'progression': 1}
                            elif t3_ge_threshold and t4_ge_threshold and t5_ge_threshold:
                                if t3_progression or t4_progression or t5_progression:
                                    return {'timepoint': base_t, 'progression': 1}

        if min_progression_points < 3:
            if (not all_previous_points_below_threshold or
                    (not t1_ge_threshold and not t2_ge_threshold)):
                base_t = 'T3'
                min_progression = min_increase_for_progression(tp_scores[base_t])
                if threshold > tp_scores[base_t] >= thresholds[0]:
                    if not first_valid_timepoint:
                        first_valid_timepoint = base_t
                    t4_progression = t4_ge_threshold and (tp_scores['T4'] - tp_scores[base_t]) >= min_progression
                    t5_progression = t5_ge_threshold and (tp_scores['T5'] - tp_scores[base_t]) >= min_progression

                    if min_progression_at_remaining_points:
                        if min_progression_at_all_points:
                            if t4_progression and t5_progression:
                                return {'timepoint': base_t, 'progression': 1}
                        elif t4_ge_threshold and t5_ge_threshold:
                            if t4_progression or t5_progression:
                                return {'timepoint': base_t, 'progression': 1}

                    else:
                        if min_progression_points == 1:
                            if t4_progression:
                                return {'timepoint': base_t, 'progression': 1}
                        elif min_progression_points == 2:
                            if min_progression_at_all_points:
                                if t4_progression and t5_progression:
                                    return {'timepoint': base_t, 'progression': 1}
                            elif t4_ge_threshold and t5_ge_threshold:
                                if t4_progression or t5_progression:
                                    return {'timepoint': base_t, 'progression': 1}

        if min_progression_points < 2:
            if (not all_previous_points_below_threshold or
                    (not t1_ge_threshold and not t2_ge_threshold and not t3_ge_threshold)):
                base_t = 'T4'
                min_progression = min_increase_for_progression(tp_scores[base_t])
                if threshold > tp_scores[base_t] >= thresholds[0]:
                    if not first_valid_timepoint:
                        first_valid_timepoint = base_t
                    t5_progression = t5_ge_threshold and (tp_scores['T5'] - tp_scores[base_t]) >= min_progression

                    if t5_progression:
                        return {'timepoint': base_t, 'progression': 1}

    return {'timepoint': first_valid_timepoint, 'progression': 0}



In [6]:
#####################################################
# Calculate Participant-based Progression
#
#####################################################

# Add the DMO data so that timepoints with missing DMOs are not considered in the calculations
dfs_list = []
for timepoint in ["T1", "T2", "T3"]:
    feature_file_path = data_dir + f"/{timepoint} Aggregated DMO Data_V7.3/cvs-{timepoint}-weekly_agg_all-21-01-2026.csv"
    temp_df = pd.read_csv(feature_file_path)
    dfs_list.append(temp_df)

features_df = pd.concat(dfs_list, ignore_index=True)
merged_data = pd.merge(data, features_df, left_on=['Local.Participant', 'visit.number'],
                       right_on=['participant_id', 'visit_type'], how='left')
merged_data = merged_data.drop(['participant_id', 'visit_type'], axis=1)

# print(f'Column NaN counts out of {len(merged_data)} T1, T2, T3 rows')
# print(merged_data[merged_data['visit.number'].isin({'T1', 'T2', 'T3'})].isna().sum())

# drop T1, T2, and T3 rows if they contain nan, which will be due to missing DMOs
count_before = merged_data["Local.Participant"].nunique()
# get indices of rows with any nan values
nan_indices = merged_data.index[merged_data.isna().any(axis=1)]
# get indices of rows with T1, T2 or T3 values
timepoint_indices = merged_data.index[merged_data['visit.number'].isin({'T1', 'T2', 'T3'})]
# Get intersection of T1, T2, T3 rows with rows containing any nan
intersection = nan_indices.intersection(timepoint_indices)
merged_data = merged_data.drop(intersection)
print(
    f"{count_before - merged_data["Local.Participant"].nunique()} participants excluded with missing DMOs at timepoints T1, T2 or T3...")

participant_data = (merged_data[['Local.Participant', 'visit.number', 'EDFSCR1L']]
                    .pivot(index='Local.Participant', columns='visit.number', values='EDFSCR1L'))

print(f"{len(participant_data)} participants read...")

count_before = len(participant_data)
# remove participants who do not have enough consecutive timepoints
participant_data = participant_data.drop(
    participant_data[
        (min_progression_points == 1 &
         ((participant_data['T1'].isnull() & participant_data['T2'].isnull()) |
          (participant_data['T2'].isnull() & participant_data['T3'].isnull()) |
          (participant_data['T3'].isnull() & participant_data['T4'].isnull()) |
          (participant_data['T4'].isnull() & participant_data['T5'].isnull())
          )) |
        (min_progression_points == 2 &
         ((participant_data['T1'].isnull() & participant_data['T2'].isnull() & participant_data['T3'].isnull()) |
          (participant_data['T2'].isnull() & participant_data['T3'].isnull() & participant_data['T4'].isnull()) |
          (participant_data['T3'].isnull() & participant_data['T4'].isnull() & participant_data['T5'].isnull())
          )) |
        (min_progression_points == 3 &
         ((participant_data['T1'].isnull() & participant_data['T2'].isnull() &
           participant_data['T3'].isnull() & participant_data['T4'].isnull()) |
          (participant_data['T2'].isnull() & participant_data['T3'].isnull() &
           participant_data['T4'].isnull() & participant_data['T5'].isnull())
          )) |
        (participant_data['T1'].isnull() & participant_data['T2'].isnull() &
         participant_data['T3'].isnull() & participant_data['T4'].isnull() &
         participant_data['T5'].isnull())].index
)
print(
    f"{count_before - len(participant_data)} participants excluded as have less than {min_progression_points + 1} consecutive timepoints...")

count_before = len(participant_data)
participant_data = participant_data[participant_data[['T1', 'T2', 'T3']].le(base_scores[-1]).any(axis='columns')]
print(
    f"{count_before - len(participant_data)} participants excluded as have T1-3 EDSS scores >= threshold ({base_scores[-1]})...")

participant_data[['timepoint', 'progression']] = participant_data.apply(
    # threshold_progress_point, thresholds=base_scores, min_score=base_scores[0],
    calc_progression, min_score=base_scores[0],
    axis='columns', result_type='expand')

count_before = len(participant_data)
participant_data = participant_data.dropna(subset=['timepoint'])
print(f"{count_before - len(participant_data)} participants excluded who have no valid progression timepoint...")

print(participant_data[participant_data['progression'] == 1].head(100))

participant_data = participant_data.merge(merged_data, how='left',
                                          left_on=['Local.Participant', 'timepoint'],
                                          right_on=['Local.Participant', 'visit.number'])

# remove processing columns
participant_data = participant_data.drop(['T1', 'T2', 'T3', 'T4', 'T5', 'timepoint'], axis=1)

if not base_scores_subset_type == "all":
    print(f"Selecting EDSS scores for subset {base_scores_subset_type}: {base_scores_subset}")
    count_before = len(participant_data)
    participant_data = participant_data[participant_data['EDFSCR1L'].isin(base_scores_subset)]
    print(f"{count_before - len(participant_data)} participants excluded who are not in the subset")

non_progressor_count = len(participant_data[participant_data['progression'] == 0])

print(f"Number of included participants = {len(participant_data)}")
print(f"Number of non-progressors = {non_progressor_count}")
print(f"Number of progressors = {len(participant_data) - non_progressor_count}")

print(pd.crosstab(participant_data['progression'], participant_data['EDFSCR1L']))


15 participants excluded with missing DMOs at timepoints T1, T2 or T3...
587 participants read...
0 participants excluded as have less than 3 consecutive timepoints...
11 participants excluded as have T1-3 EDSS scores >= threshold (6.5)...
2 participants excluded who have no valid progression timepoint...
visit.number        T1   T2   T3   T4   T5 timepoint  progression
Local.Participant                                                
10376              6.0  6.0  5.5  6.0  6.0        T3          1.0
10382              4.0  6.0  6.0  5.0  NaN        T1          1.0
10384              NaN  6.0  5.0  6.0  6.0        T3          1.0
10385              NaN  4.0  4.5  6.0  6.0        T3          1.0
10396              6.0  6.0  6.5  6.5  6.5        T2          1.0
10425              6.0  6.0  6.0  6.5  6.5        T3          1.0
21149              3.0  4.0  4.0  NaN  NaN        T1          1.0
21159              6.0  6.5  6.5  6.5  6.5        T1          1.0
21161              3.5  3.0  4.0 

In [7]:
#####################################################
# Prepare Training Data
#
#####################################################

data = participant_data

# drop irrelevant or 'cheating' columns
y_data = data["progression"].reset_index(drop=True)  # can be data["progression"] or data["multiclass_progression"]
labels_X_all = data.reset_index(drop=True)

non_dmo_features = ['EDFSCR1L', 'age', 'gender', 'time_since_diag', 'time_since_onset']
if drop_DMO_features:
    X_data = data[non_dmo_features].reset_index(drop=True)
else:
    features_to_drop = ["progression", "Local.Participant", "visit.number"]
    if drop_non_DMO_features:
        features_to_drop += non_dmo_features
    X_data = data.drop(features_to_drop, axis=1)

if (drop_features_as_predictors):
    X_data = X_data.drop(drop_features_as_predictors, axis=1)
if (only_include_features):
    X_data = X_data[only_include_features]


In [8]:
#####################################################
# Data normalisation
#
#####################################################

# Data normalisation (Note the impact of this needs to be tested)
# e.g. StandardScaler, MinMaxScaler, RobustScaler
# If columns have different (e.g. skewed) distributions that may need different normalisations
# e.g. Log normalisation
# Note that StandardScaler is generally preferred,
# however combined with small data will hang SVC, while MinMaxScaler works

# moved scaler to inside function to prevent re-fitting

if show_data_distributions:
    for i, column in enumerate(X_data.columns):
        if X_data[column].dtype == 'float64':
            sns.displot(data=X_data, x=column, kind="kde")
        else:
            sns.displot(X_data, x=column)



In [ ]:
###################################################################
# Utility functions for cross-validation and results
###################################################################

def do_cv(model, X_data, y_data, n_splits=cv_folds, random_state=random_state, quiet=False):
    # requires: set_config(transform_output="pandas")
    # transformer = ColumnTransformer([('standard_scaler', ss, feature_cols)],
    #                                 remainder='passthrough',
    #                                 verbose_feature_names_out=False)

    combined_cm = None
    combined_y_test = None
    combined_y_pred = None
    kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    for i, (train_split_indices, test_split_indices) in enumerate(kf.split(X_data, y_data)):

        X_train_split = X_data.iloc[train_split_indices]
        X_test_split = X_data.iloc[test_split_indices]
        y_train_split = y_data.iloc[train_split_indices]
        y_test_split = y_data.iloc[test_split_indices]

        scaler = MinMaxScaler()
        X_train_scaled = scaler.fit_transform(X_train_split)
        X_test_scaled = scaler.transform(X_test_split)

        # sample weighting
        sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_split)

        # we need this to deal with the KNN not accepting sample weights
        try:
            model.fit(X_train_scaled, y_train_split, sample_weight=sample_weights)
        except Exception as e:
            if not quiet:
                warning = RuntimeWarning(*e.args)
                warning.with_traceback(e.__traceback__)
                warnings.warn(warning)
                warnings.warn(
                    "Sample weighting not accepted, training without and using random oversampling to achieve sample weighting... ")
            ros = RandomOverSampler(random_state=random_state)
            X_train_resampled, y_train_resampled = ros.fit_resample(X_train_scaled, y_train_split)
            model.fit(X_train_resampled, y_train_resampled)

        y_pred = model.predict(X_test_scaled)

        if not quiet:
            print(f"Fold {i}")
            print(f"accuracy: {accuracy_score(y_test_split, y_pred):.2f}")
            print(f"precision_macro: {precision_score(y_test_split, y_pred, average='macro', zero_division=0):.2f}")
            print(f"recall_macro: {recall_score(y_test_split, y_pred, average='macro', zero_division=0):.2f}")
            print(f"f1_macro: {f1_score(y_test_split, y_pred, average='macro', zero_division=0):.2f}")

        cm = confusion_matrix(y_test_split, y_pred)
        # turn y_pred into a series
        y_pred_series = pd.Series(y_pred, index=y_test_split.index)
        combined_cm = cm if combined_cm is None else combined_cm + cm
        combined_y_test = y_test_split if combined_y_test is None else pd.concat([combined_y_test, y_test_split])
        combined_y_pred = y_pred_series if combined_y_pred is None else pd.concat([combined_y_pred, y_pred_series])

    return combined_cm, combined_y_test, combined_y_pred


# ToDo make generic, i.e. able to handle different sized CM
def display_cm(cm, fig_name=None):
    # Pretty print confusion matrix
    ax = plt.subplot()

    cm = pd.DataFrame(cm, range(2), range(2))
    sns.heatmap(cm, annot=True, cmap="Blues", ax=ax, fmt='d')

    ax.set_xlabel('Predicted labels')
    ax.set_ylabel('True labels')
    ax.xaxis.set_ticklabels(['0', '1'])
    ax.yaxis.set_ticklabels(['0', '1'])
    if not fig_name == None:
        plt.title(fig_name)
        # plt.savefig(fig_save_dir + fig_name)
    plt.show()


def show_evaluation(model, X_test, y_test, fig_name=None):
    y_pred = model.predict(X_test)

    print(f"accuracy: {accuracy_score(y_test, y_pred):.2f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    cm = confusion_matrix(y_test, y_pred)

    display_cm(cm, fig_name)


def get_metrics_from_cm(cm, type="micro", dp=3):
    tn, fp, fn, tp = cm.ravel()

    accuracy = (tp + tn) / (tp + tn + fp + fn)
    if (type == "micro"):
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    else:
        pos_precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        neg_precision = tn / (tn + fn) if (tn + fn) > 0 else 0
        precision = (pos_precision + neg_precision) / 2.0
        pos_recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        neg_recall = tn / (tn + fp) if (tn + fp) > 0 else 0
        recall = (pos_recall + neg_recall) / 2.0

    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return {
        "type": type,
        "accuracy": round(accuracy, dp),
        "precision": round(precision, dp),
        "recall": round(recall, dp),
        "f1": round(f1_score, dp)
    }


def get_metrics_from_prediction(actual, predicted):
    return {
        "accuracy": accuracy_score(actual, predicted),
        "precision": precision_score(actual, predicted),
        "recall": recall_score(actual, predicted),
        "f1": f1_score(actual, predicted),
        "precision_macro": precision_score(actual, predicted, average='macro', zero_division=0),
        "recall_macro": recall_score(actual, predicted, average='macro', zero_division=0),
        "f1_macro": f1_score(actual, predicted, average='macro', zero_division=0),
    }


In [9]:
###################################################################
# Hyper Parameter Tuning (HPT)
#
# Uses GridSearchCV to search around the default settings
# Uses RepeatedStratifiedKFold with a low number of splits (2) and high repetitions (5/10) to ensure HPT is generalising
###################################################################

param_search_stop = False


def param_search_time_out():
    global param_search_stop
    param_search_stop = True


def set_params(model, params):
    for k, v in params.items():
        model.set_params(**{k: v})


def get_params_score(model, X_data, y_data, n_splits, n_repeats):
    scores = []
    cm, y_test, y_pred, fold_metrics = do_cv(
        model, X_data, y_data, n_splits=n_splits, n_repeats=n_repeats, quiet=True)
    macro_results = get_metrics_from_prediction(y_test, y_pred)
    scores.append(macro_results[scoring])

    return sum(scores) / len(scores)


def param_score(create_model, X_data, y_data, n_splits, n_repeats,
                all_param_combinations, to_process, i, start):
    if param_search_stop:
        return None

    params = all_param_combinations.pop(random.randint(0, len(all_param_combinations) - 1))

    model = create_model()
    # set test params
    set_params(model, params)

    score = get_params_score(model, X_data, y_data, n_splits, n_repeats)

    if i == 10 or i == 100 or i == 1000:
        elapsed = timer() - start
        eta = elapsed * (to_process - i) / i

        print(f'Done {i} of {to_process} in {elapsed:.1f}s: ETA = {eta:.1f}s')
    return {
        'params': params,
        'score': score
    }


def param_search(create_model, parameters, X_data, y_data,
                 n_splits=n_splits, n_repeats=n_repeats,
                 type=hpt_search_type, limit=hpt_search_limit):
    if type not in ['grid', 'random_proportion', 'random_time', 'random_n_params']:
        raise Exception(f"Unknown param search type: {type}")

    model = create_model()

    equal_best_params = []
    best_params = {}
    best_score = -1
    all_param_combinations = list(ParameterGrid(parameters))

    all_default_params = model.get_params()
    default_score = get_params_score(model, X_data, y_data, n_splits, n_repeats)

    n_combinations = len(all_param_combinations)
    to_process = n_combinations

    if type == 'random_proportion':
        if limit <= 0 or limit >= 1.0:
            raise Exception(f"Random proportion search limit must be between 0 and 1, found {limit}")
        to_process = 1 + int(n_combinations * limit)
        print(
            f'{get_model_name(model)}: random search optimisation of {scoring} on {to_process} of {n_combinations} combinations')
    elif type == 'random_time':
        print(
            f'{get_model_name(model)}: random search optimisation of {scoring} for {limit} seconds on {n_combinations} combinations')
        Timer(limit, param_search_time_out).start()
    elif type == 'random_n_params':
        if limit <= 0:
            raise Exception(f"Random params search limit must be greater than 0, found {limit}")
        to_process = limit if limit < n_combinations else n_combinations
        print(
            f'{get_model_name(model)}: random search optimisation of {scoring} on {to_process} of {n_combinations} combinations')

    else:  # type == 'grid'
        print(f'{get_model_name(model)}: grid search optimisation of {scoring} on {n_combinations} combinations')

    global param_search_stop
    param_search_stop = False
    start = timer()

    results = Parallel(n_jobs=hpt_n_jobs)(
        delayed(param_score)(create_model, X_data, y_data, n_splits, n_repeats,
                             all_param_combinations, to_process, i, start)
        for i in range(1, to_process))

    for result in results:
        if (not result == None):
            score = result['score']
            params = result['params']
            if score > best_score:
                best_score = score
                best_params = params
                equal_best_params.clear()
            elif score == best_score:
                equal_best_params.append(params)

    print(f'Done {n_combinations - len(all_param_combinations)} in {timer() - start:.1f}s')

    # get default values for params sent to hpt
    default_params = {key: all_default_params[key] for key in best_params.keys()}
    print(f'default: {default_score:.3f} = {default_params}')
    print(f'best: {best_score:.3f} = {best_params}')
    if len(equal_best_params) > 0:
        print(f'{len(equal_best_params)} equal best: {equal_best_params}')

    set_params(model, all_default_params)
    set_params(model, best_params)
    cm, y_test, y_pred = do_cv(
        model, X_data, y_data, n_splits=n_splits, quiet=True)
    print(cm)
    print(get_metrics_from_prediction(y_test, y_pred))

    return best_score, best_params, equal_best_params


In [10]:
###################################################################
# Test Hyper Parameter Tuning
#
###################################################################

x = []
accuracy = []
micro_precision = []
micro_recall = []
micro_f1 = []
macro_precision = []
macro_recall = []
macro_f1 = []


def clear_results():
    x.clear()
    accuracy.clear()
    micro_precision.clear()
    micro_recall.clear()
    micro_f1.clear()
    macro_precision.clear()
    macro_recall.clear()
    macro_f1.clear()


def add_results(model, i, print_model):
    cm, y_test, y_pred = do_cv(model, X_data, y_data, quiet=True)
    metrics = get_metrics_from_prediction(y_test, y_pred)
    x.append(i)
    accuracy.append(metrics["accuracy"])
    micro_precision.append(metrics["precision"])
    micro_recall.append(metrics["recall"])
    micro_f1.append(metrics["f1"])
    macro_precision.append(metrics["precision_macro"])
    macro_recall.append(metrics["recall_macro"])
    macro_f1.append(metrics["f1_macro"])

    if print_model:
        print(cm)
        print(get_metrics_from_cm(cm, "macro"))

        if (hasattr(model, 'best_params_')):
            print(model.best_params_)
            print(model.estimator.get_params())
        else:
            print(model.get_params())
        macro_results = get_metrics_from_prediction(y_test, y_pred)
        print(macro_results)
        print(classification_report(y_test, y_pred))


def show_results():
    plt.plot(x, accuracy, label="accuracy")
    plt.plot(x, macro_precision, label="macro precision")
    plt.plot(x, macro_recall, label="macro recall")
    plt.plot(x, macro_f1, label="macro f1")
    plt.legend(loc='upper left')
    plt.show()


In [16]:
###################################################################
# Test SVM Hyper Parameter Tuning
#
###################################################################

default_svc_parameters = [{'C': [0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8],
                           'degree': [3],
                           'gamma': ['scale', 'auto', 0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05, 0.1, 0.5],
                           'kernel': ['rbf']}]
Cs = [0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8]
gammas = ['scale', 'auto', 0.00001, 0.00005, 0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05, 0.1, 0.5]
degrees = [1, 2, 3, 4, 5]
svc_parameters = [{'C': Cs, 'kernel': ['linear']},
                  {'C': Cs, 'kernel': ['rbf', 'sigmoid'], 'gamma': gammas},
                  {'C': Cs, 'kernel': ['poly'], 'gamma': gammas, 'degree': degrees}
                  ]
svc_rbf_parameters = [{'C': [x * 0.1 for x in range(20)], 'kernel': ['rbf'], 'gamma': ['scale']}]


svc_bayes_parameters = [{
    'C': Real(1e-6, 1e+6, prior='log-uniform'),  # Regularization parameter
    'gamma': Real(1e-6, 1e+1, prior='log-uniform'),  # Kernel coefficient
    'kernel': Categorical(['rbf', 'sigmoid']),  # Type of kernel
    'degree': Integer(1, 5)  # Degree for poly kernel
}]

svc_bayes_parameters2 = [
    {'C': Real(1e-6, 1e+6, prior='log-uniform'), 'kernel': ['linear']},
    {'C': Real(1e-6, 1e+6, prior='log-uniform'), 'kernel': ['rbf', 'sigmoid'],
     'gamma': Real(1e-6, 1e+1, prior='log-uniform')},
    {'C': Real(1e-6, 1e+6, prior='log-uniform'), 'kernel': ['poly'],
     'gamma': Real(1e-6, 1e+1, prior='log-uniform'), 'degree': Integer(1, 5)},
]


def test_svm_BayesSearchCV_hpt_n_iter(max_n_iter, iter_multiplier):
    clear_results()
    print(
        f'Testing BayesSearchCV optimisation of {scoring}, with {max_n_iter * iter_multiplier} iterations on SVC model')

    for hpt_n_iter in range(1, max_n_iter + 1):
        start = timer()
        model = BayesSearchCV(
            SVC(random_state=random_state),
            svc_parameters,
            n_iter=hpt_n_iter * iter_multiplier,
            random_state=random_state,
            verbose=hpt_verbose,
            n_jobs=hpt_n_jobs,
            cv=RepeatedStratifiedKFold(n_splits=5, n_repeats=1, random_state=random_state),
            scoring=scoring
        )
        add_results(model, hpt_n_iter * iter_multiplier, hpt_n_iter == max_n_iter)
        end = timer()
        print(f'{hpt_n_iter * iter_multiplier} iterations in {end - start:.2f} seconds')

    show_results()
    print(model.best_params_)
    cm, y_test, y_pred = do_cv(model.best_estimator_, X_data, y_data, quiet=True)
    macro_results = get_metrics_from_prediction(y_test, y_pred)
    print(macro_results)

def test_svm_RandomizedSearchCV_hpt_n_iter(max_n_iter, iter_multiplier):
    clear_results()
    print(
        f'Testing RandomizedSearchCV optimisation of {scoring}, with {max_n_iter * iter_multiplier} iterations on SVC model')
    for hpt_n_iter in range(1, max_n_iter + 1):
        start = timer()
        model = RandomizedSearchCV(
            SVC(random_state=random_state),
            svc_parameters,
            n_iter=hpt_n_iter * iter_multiplier,
            random_state=random_state,
            verbose=hpt_verbose,
            n_jobs=hpt_n_jobs,
            cv=RepeatedStratifiedKFold(n_splits=2, n_repeats=2, random_state=random_state),
            scoring=scoring
        )
        add_results(model, hpt_n_iter * iter_multiplier, hpt_n_iter == max_n_iter)
        end = timer()
        print(f'{hpt_n_iter * iter_multiplier} iterations in {end - start:.2f} seconds')

    show_results()
    cm, y_test, y_pred = do_cv(model.best_estimator_, X_data, y_data, quiet=True)
    macro_results = get_metrics_from_prediction(y_test, y_pred)
    print(macro_results)


def test_svm_GridSearchCV(model):
    print(f'Testing GridSearchCV optimisation of {scoring} on SVC model')
    start = timer()
    model = GridSearchCV(
        model,
        svc_rbf_parameters,
        verbose=hpt_verbose,
        n_jobs=hpt_n_jobs,
        cv=RepeatedStratifiedKFold(n_splits=2, n_repeats=5, random_state=random_state),
        scoring=scoring,
    )
    model.fit(X_data, y_data)
    # cm, y_test, y_pred = do_cv(model, X_data, y_data, quiet=True)
    end = timer()
    print(f'{end - start:.2f} seconds')

    print("Best Params: ", model.best_params_)
    print("Best Score: ", model.best_score_)
    # macro_results = get_metrics_from_prediction(y_test, y_pred)
    # print(macro_results)
    results = model.cv_results_
    results = pd.concat([pd.DataFrame(results["params"]),
                         pd.DataFrame(results["mean_test_score"],
                                      columns=[scoring])],
                        axis=1)
    results = results.sort_values([scoring], ascending=[False])
    # with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(results)

    cm, y_test, y_pred = do_cv(model.best_estimator_, X_data, y_data, quiet=True)
    macro_results = get_metrics_from_prediction(y_test, y_pred)
    print(macro_results)


if (do_svm_hpt_test):
    clear_results()
    add_results(SVC(random_state=random_state), 0, True)

    # test_svm_BayesSearchCV_hpt_n_iter(5, 2)
    # test_svm_RandomizedSearchCV_hpt_n_iter(15, 2)
    test_svm_GridSearchCV(model)

[[319 203]
 [ 12  40]]
{'type': 'macro', 'accuracy': 0.625, 'precision': 0.564, 'recall': 0.69, 'f1': 0.621}
{'C': 1.0, 'break_ties': False, 'cache_size': 200, 'class_weight': None, 'coef0': 0.0, 'decision_function_shape': 'ovr', 'degree': 3, 'gamma': 'scale', 'kernel': 'rbf', 'max_iter': -1, 'probability': False, 'random_state': 42, 'shrinking': True, 'tol': 0.001, 'verbose': False}
{'accuracy': 0.6254355400696864, 'precision': 0.1646090534979424, 'recall': 0.7692307692307693, 'f1': 0.2711864406779661, 'precision_macro': 0.5641776385314485, 'recall_macro': 0.6901709401709402, 'f1_macro': 0.5095674290142468}
              precision    recall  f1-score   support

         0.0       0.96      0.61      0.75       522
         1.0       0.16      0.77      0.27        52

    accuracy                           0.63       574
   macro avg       0.56      0.69      0.51       574
weighted avg       0.89      0.63      0.70       574

Testing GridSearchCV optimisation of recall_macro on SVC 

ValueError: Invalid parameter 'C' for estimator Pipeline(steps=[SVC(random_state=42)]). Valid parameters are: ['memory', 'steps', 'transform_input', 'verbose'].

In [12]:
###################################################################
# Test KNN Hyper Parameter Tuning
#
###################################################################

neighbors = [1, 2, 3, 4, 5]
ps = [1, 2]
leaf_sizes = [5, 10, 20, 30, 40, 50]
metrics = ['euclidean']  #, 'braycurtis', 'canberra', 'chebyshev', 'cityblock', 'correlation', 'cosine', 'sqeuclidean']
kdtree_metrics = ['euclidean']  #, 'cityblock', 'chebyshev']
balltree_metrics = ['euclidean']  #, 'cityblock', 'chebyshev', 'canberra', 'braycurtis']

knn_balanced = False

def get_knn(balanced):
    return Pipeline([
        ('sampler', RandomOverSampler(random_state=random_state)),
        ('classifier', KNeighborsClassifier())
    ]) if balanced else KNeighborsClassifier()


def get_knn_parameters(balanced):
    prefix = 'classifier__' if balanced else ''
    return [
        {prefix + 'n_neighbors': neighbors, prefix + 'weights': ['uniform'],
         prefix + 'algorithm': ['brute']},
        {prefix + 'n_neighbors': neighbors, prefix + 'weights': ['uniform'],
         prefix + 'algorithm': ['ball_tree', 'kd_tree'], prefix + 'leaf_size': leaf_sizes},
        {prefix + 'n_neighbors': neighbors, prefix + 'weights': ['distance'],
         prefix + 'metric': ['minkowski'], prefix + 'p': ps,
         prefix + 'algorithm': ['brute']},
        {prefix + 'n_neighbors': neighbors, prefix + 'weights': ['distance'],
         prefix + 'metric': ['minkowski'], prefix + 'p': ps,
         prefix + 'algorithm': ['ball_tree', 'kd_tree'], prefix + 'leaf_size': leaf_sizes},
        {prefix + 'n_neighbors': neighbors, prefix + 'weights': ['distance'],
         prefix + 'metric': metrics, prefix + 'algorithm': ['auto', 'brute']},
        {prefix + 'n_neighbors': neighbors, prefix + 'weights': ['distance'],
         prefix + 'metric': ['minkowski'], prefix + 'p': ps,
         prefix + 'algorithm': ['ball_tree'], prefix + 'leaf_size': leaf_sizes},
        {prefix + 'n_neighbors': neighbors, prefix + 'weights': ['distance'],
         prefix + 'metric': balltree_metrics, prefix + 'algorithm': ['ball_tree'],
         prefix + 'leaf_size': leaf_sizes},
        {prefix + 'n_neighbors': neighbors, prefix + 'weights': ['distance'],
         prefix + 'metric': ['minkowski'], prefix + 'p': ps,
         prefix + 'algorithm': ['kd_tree'], prefix + 'leaf_size': leaf_sizes},
        {prefix + 'n_neighbors': neighbors, prefix + 'weights': ['distance'],
         prefix + 'metric': kdtree_metrics, prefix + 'algorithm': ['kd_tree'],
         prefix + 'leaf_size': leaf_sizes}
    ]


knn_bayes_parameters = [
    {
        'n_neighbors': Integer(1, 50),
        'weights': Categorical(['uniform', 'distance']),
        'algorithm': Categorical(['auto', 'ball_tree', 'kd_tree', 'brute']),
        'leaf_size': Integer(10, 50),
        'p': Integer(1, 2)  # 1 = Manhattan, 2 = Euclidean distance
    }]


def test_knn_BayesSearchCV_hpt_n_iter(max_n_iter, iter_multiplier):
    clear_results()
    print(
        f'Testing BayesSearchCV optimisation of {scoring}, with {max_n_iter * iter_multiplier} iterations on KNN model')

    for hpt_n_iter in range(1, max_n_iter + 1):
        start = timer()
        model = BayesSearchCV(
            get_knn(knn_balanced),
            get_knn_parameters(knn_balanced),
            n_iter=hpt_n_iter,
            random_state=random_state,
            verbose=hpt_verbose,
            n_jobs=hpt_n_jobs,
            cv=hpt_cv,
            scoring=scoring
        )
        add_results(model, hpt_n_iter * iter_multiplier, hpt_n_iter == max_n_iter)
        end = timer()
        print(f'{hpt_n_iter * iter_multiplier} iterations in {end - start:.2f} seconds')
    show_results()


def test_knn_RandomizedSearchCV_hpt_n_iter(max_n_iter, iter_multiplier):
    clear_results()
    print(
        f'Testing RandomizedSearchCV optimisation of {scoring}, with {max_n_iter * iter_multiplier} iterations on KNN model')
    for hpt_n_iter in range(1, max_n_iter + 1):
        start = timer()
        model = RandomizedSearchCV(
            get_knn(knn_balanced),
            get_knn_parameters(knn_balanced),
            n_iter=hpt_n_iter * iter_multiplier,
            random_state=random_state,
            verbose=hpt_verbose,
            n_jobs=hpt_n_jobs,
            cv=hpt_cv,
            scoring=scoring
        )
        add_results(model, hpt_n_iter * iter_multiplier, hpt_n_iter == max_n_iter)
        end = timer()
        print(f'{hpt_n_iter * iter_multiplier} iterations in {end - start:.2f} seconds')

    show_results()


def test_knn_GridSearchCV():
    print(f'Testing GridSearchCV optimisation of {scoring} on KNN model')
    start = timer()
    model = GridSearchCV(
        get_knn(knn_balanced),
        get_knn_parameters(knn_balanced),
        verbose=hpt_verbose,
        n_jobs=hpt_n_jobs,
        cv=hpt_cv,
        scoring=scoring
    )
    # model.fit(X_data, y_data)
    cm, y_test, y_pred = do_cv(model, X_data, y_data, quiet=True)
    end = timer()
    print(f'{end - start:.2f} seconds')

    print(model.best_params_)
    print(model.estimator.get_params())
    macro_results = get_metrics_from_prediction(y_test, y_pred)
    print(macro_results)
    results = model.cv_results_
    results = pd.concat([pd.DataFrame(results["params"]),
                         pd.DataFrame(results["mean_test_score"],
                                      columns=[scoring])],
                        axis=1)
    if knn_balanced:
        results.rename(columns={'classifier__algorithm': 'algorithm', 'classifier__n_neighbors': 'n_neighbors',
                                "classifier__weights": 'weights', 'classifier__leaf_size': 'leaf_size',
                                'classifier__metric': 'metric', 'classifier__p': 'p'}, inplace=True)
    results = results.sort_values([scoring, 'algorithm', 'n_neighbors', 'weights', 'metric', 'leaf_size', 'p'],
                                  ascending=[False, True, True, True, True, True, True])
    with pd.option_context('display.width', 1000):  #, 'display.max_rows', None, 'display.max_columns', None):
        print(results)
    print(classification_report(y_data, y_pred))


if (do_knn_hpt_test):
    test_knn_BayesSearchCV_hpt_n_iter(10, 2)
    test_knn_RandomizedSearchCV_hpt_n_iter(10, 2)
    # test_knn_GridSearchCV()


Testing BayesSearchCV optimisation of recall_macro, with 20 iterations on KNN model
2 iterations in 6.12 seconds
4 iterations in 6.73 seconds
6 iterations in 7.30 seconds
8 iterations in 7.95 seconds
10 iterations in 8.63 seconds
12 iterations in 9.21 seconds
14 iterations in 10.11 seconds


KeyboardInterrupt: 

In [ ]:
###################################################################
# Test MLP Hyper Parameter Tuning
#
###################################################################

mlp_balanced = False


def get_mlp(balanced):
    mlp = MLPClassifier(random_state=random_state, max_iter=1000)
    return Pipeline([
        ('sampler', RandomOverSampler(random_state=random_state)),
        ('classifier', mlp)
    ]) if balanced else mlp


def get_mlp_parameters(balanced):
    prefix = 'classifier__' if balanced else ''
    return [
        {prefix + 'hidden_layer_sizes': [50, 100],
         prefix + 'activation': ['logistic', 'tanh', 'relu'],
         prefix + 'solver': ['lbfgs', 'sgd', 'adam'],
         prefix + 'alpha': [0.0001, 0.001, 0.01, 0.05],
         prefix + 'learning_rate': ['constant', 'invscaling', 'adaptive'],
         prefix + 'learning_rate_init': [0.0001, 0.001, 0.01, 0.1]  # Initial learning rate
         }
    ]


def test_mlp_BayesSearchCV_hpt_n_iter(max_n_iter, iter_multiplier):
    clear_results()
    print(
        f'Testing BayesSearchCV optimisation of {scoring}, with {max_n_iter * iter_multiplier} iterations on MLP model')

    for hpt_n_iter in range(1, max_n_iter + 1):
        start = timer()
        model = BayesSearchCV(
            get_mlp(mlp_balanced),
            get_mlp_parameters(mlp_balanced),
            n_iter=hpt_n_iter,
            random_state=random_state,
            verbose=hpt_verbose,
            n_jobs=hpt_n_jobs,
            cv=hpt_cv,
            scoring=scoring
        )
        add_results(model, hpt_n_iter * iter_multiplier, hpt_n_iter == max_n_iter)
        end = timer()
        print(f'{hpt_n_iter * iter_multiplier} iterations in {end - start:.2f} seconds')
    show_results()


def test_mlp_RandomizedSearchCV_hpt_n_iter(max_n_iter, iter_multiplier):
    clear_results()
    print(
        f'Testing RandomizedSearchCV optimisation of {scoring}, with {max_n_iter * iter_multiplier} iterations on MLP model')
    for hpt_n_iter in range(1, max_n_iter + 1):
        start = timer()
        model = RandomizedSearchCV(
            get_mlp(mlp_balanced),
            get_mlp_parameters(mlp_balanced),
            n_iter=hpt_n_iter * iter_multiplier,
            random_state=random_state,
            verbose=hpt_verbose,
            n_jobs=hpt_n_jobs,
            cv=hpt_cv,
            scoring=scoring
        )
        add_results(model, hpt_n_iter * iter_multiplier, hpt_n_iter == max_n_iter)
        end = timer()
        print(f'{hpt_n_iter * iter_multiplier} iterations in {end - start:.2f} seconds')

    show_results()


def test_mlp_GridSearchCV():
    print(f'Testing GridSearchCV optimisation of {scoring} on MLP model')
    start = timer()
    model = GridSearchCV(
        get_mlp(mlp_balanced),
        get_mlp_parameters(mlp_balanced),
        verbose=hpt_verbose,
        n_jobs=hpt_n_jobs,
        cv=hpt_cv,
        scoring=scoring
    )
    # model.fit(X_data, y_data)
    cm, y_test, y_pred = do_cv(model, X_data, y_data, quiet=True)
    end = timer()
    print(f'{end - start:.2f} seconds')

    print(model.best_params_)
    print(model.estimator.get_params())
    macro_results = get_metrics_from_prediction(y_test, y_pred)
    print(macro_results)
    results = model.cv_results_
    results = pd.concat([pd.DataFrame(results["params"]),
                         pd.DataFrame(results["mean_test_score"],
                                      columns=[scoring])],
                        axis=1)
    if mlp_balanced:
        results.rename(
            columns={'classifier__hidden_layer_sizes': 'hidden_layer_sizes', 'classifier__activation': 'activation',
                     "classifier__solver": 'solver', 'classifier__alpha': 'alpha',
                     'classifier__learning_rate': 'learning_rate',
                     'classifier__learning_rate_init': 'learning_rate_init'}, inplace=True)

    results = results.sort_values(
        [scoring, 'hidden_layer_sizes', 'activation', 'solver', 'alpha', 'learning_rate', 'learning_rate_init'],
        ascending=[False, True, True, True, True, True, True])
    with pd.option_context('display.width', 1000):  #, 'display.max_rows', None, 'display.max_columns', None):
        print(results)
    print(classification_report(y_data, y_pred))


if (do_mlp_hpt_test):
    test_mlp_BayesSearchCV_hpt_n_iter(5, 1)
    # test_mlp_RandomizedSearchCV_hpt_n_iter(20, 2)
    # test_mlp_GridSearchCV()


In [ ]:
###################################################################
# Test xgb Hyper Parameter Tuning
#
###################################################################

xgb_class_weight = None


def get_xgb():
    return XGBClassifier(random_state=random_state, use_label_encoder=False, eval_metric='logloss',
                         class_weight=xgb_class_weight)


def get_xgb_parameters():
    return [
        {
            'n_estimators': [50, 100, 200, 300],
            'max_depth': [3, 5, 7, 9, 11, 15],
            'learning_rate': [0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1],
            'subsample': [0.5, 1],  # Fraction of samples used for fitting
            'colsample_bytree': [0.3, 0.5, 0.7],  # Fraction of features used per tree
            'gamma': [0.001, 0.01, 0.1, 0.2, 0.3],  # Min loss reduction for split
            'reg_alpha': [0.00001],  # L1 regularization
            'reg_lambda': [0.00001],  # L2 regularization
            'min_child_weight': [1, 3, 5, 7]
        }
    ]


def test_xgb_BayesSearchCV_hpt_n_iter(max_n_iter, iter_multiplier):
    clear_results()
    print(
        f'Testing BayesSearchCV optimisation of {scoring}, with {max_n_iter * iter_multiplier} iterations on xgb model')

    for hpt_n_iter in range(1, max_n_iter + 1):
        start = timer()
        model = BayesSearchCV(
            get_xgb(),
            get_xgb_parameters(),
            n_iter=hpt_n_iter,
            random_state=random_state,
            verbose=hpt_verbose,
            n_jobs=hpt_n_jobs,
            cv=hpt_cv,
            scoring=scoring
        )
        add_results(model, hpt_n_iter * iter_multiplier, hpt_n_iter == max_n_iter)
        end = timer()
        print(f'{hpt_n_iter * iter_multiplier} iterations in {end - start:.2f} seconds')
    show_results()


def test_xgb_RandomizedSearchCV_hpt_n_iter(max_n_iter, iter_multiplier):
    clear_results()
    print(
        f'Testing RandomizedSearchCV optimisation of {scoring}, with {max_n_iter * iter_multiplier} iterations on xgb model')
    for hpt_n_iter in range(1, max_n_iter + 1):
        start = timer()
        model = RandomizedSearchCV(
            get_xgb(),
            get_xgb_parameters(),
            n_iter=hpt_n_iter * iter_multiplier,
            random_state=random_state,
            verbose=hpt_verbose,
            n_jobs=hpt_n_jobs,
            cv=hpt_cv,
            scoring=scoring
        )
        add_results(model, hpt_n_iter * iter_multiplier, hpt_n_iter == max_n_iter)
        end = timer()
        print(f'{hpt_n_iter * iter_multiplier} iterations in {end - start:.2f} seconds')

    show_results()


def test_xgb_GridSearchCV():
    print(f'Testing GridSearchCV optimisation of {scoring} on xgb model')
    start = timer()
    model = GridSearchCV(
        get_xgb(),
        get_xgb_parameters(),
        verbose=hpt_verbose,
        n_jobs=hpt_n_jobs,
        cv=hpt_cv,
        scoring=scoring
    )
    # model.fit(X_data, y_data)
    cm, y_test, y_pred = do_cv(model, X_data, y_data, quiet=True)
    end = timer()
    print(f'{end - start:.2f} seconds')

    print(model.best_params_)
    print(model.estimator.get_params())
    macro_results = get_metrics_from_prediction(y_test, y_pred)
    print(macro_results)
    results = model.cv_results_
    results = pd.concat([pd.DataFrame(results["params"]),
                         pd.DataFrame(results["mean_test_score"],
                                      columns=[scoring])],
                        axis=1)

    results = results.sort_values(
        [scoring, 'n_estimators', 'max_depth', 'learning_rate', 'subsample', 'colsample_bytree', 'gamma', 'reg_alpha',
         'reg_lambda'],
        ascending=[False, True, True, True, True, True, True, True, True])

    with pd.option_context('display.width', 1000):  #, 'display.max_rows', None, 'display.max_columns', None):
        print(results)
    print(classification_report(y_data, y_pred))


if (do_xgb_hpt_test):
    test_xgb_BayesSearchCV_hpt_n_iter(5, 1)
    # test_xgb_RandomizedSearchCV_hpt_n_iter(20, 2)
    # test_xgb_GridSearchCV()
